# 예제 franka_ex08: FR3 경로 제약조건 — Orientation Path Constraint

Franka FR3 끝단의 **방향(아래)** 을 *경로 내내* 유지하면서 여러 목표 지점을 순회.
"물컵 운반" 시나리오 — 가는 도중에도 손목이 뒤집히면 안 된다.

## 이 노트북이 강조하는 것 — `path_constraints` 의 도입

ex03~07 의 모든 plan 은 `goal_constraints` 만 채웠다 (시작 / 끝 조건). ex08 은 처음으로
`MotionPlanRequest.path_constraints` 에도 제약을 넣는다 — **경로 도중에도 만족해야 할
조건** 이다.

| Constraints | 언제 만족 | 본 예제에서 |
|---|---|---|
| `goal_constraints` | 도착점에서만 | 위치 + 방향 |
| `path_constraints` | 경로 내내 | **끝단 방향 (아래)** 유지 |

`OrientationConstraint` 의 `absolute_x/y/z_axis_tolerance` 는 라디안 단위 회전 허용량.
yaw 자유도를 풀고 싶으면 `tol_z = π` 정도를 주면 된다.

운영 노하우 — **제약 하 플래닝은 일반 plan 보다 어려우므로**
`num_planning_attempts` / `allowed_planning_time` 을 늘려야 (예: 5→15, 10s→30s).

## 이전 예제와의 관계

ex07 의 `plan_viz_execute()` 패턴 (plan-only → FK 미리보기 → execute) 을 그대로 재사용하고,
**`path_quat` 옵셔널 인자 한 개** 로 path constraint 를 켜고 끈다 — 같은 함수로 제약 하 / 없음
비교가 가능. RViz 에서 EE 경로 색이 달라져 (제약 하 = 청록, 없음 = 회색) 차이가 한눈에.

## 노트북 구성
1. **로봇 상수**
2. **핵심 — `plan_viz_execute(..., path_quat=...)` 워크플로** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 노드, 클라이언트(FK 포함), SRDF, Pose / MoveGroup / 마커 헬퍼
4. **시나리오** — 시작점 → A → B → C → 비교(제약 없이 B) → ready

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/constraint_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex08_constraints.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 상수

In [2]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/constraint_markers'

## 2. 핵심 — `path_constraints` 를 얹은 `plan_viz_execute`

이 노트북이 가장 먼저 정의해야 하는 함수들. 다섯이 한 묶음으로 본 예제의 본질을 이룬다.

- `make_orientation_path_constraint()` — 경로 내내 유지할 방향 제약 빌더
- `plan_to_pose_goal()` — `path_quat` 옵셔널 인자로 path_constraints 토글
- `trajectory_to_ee_path()` / `publish_ee_path()` / `execute_trajectory()` — ex07 과 동일
- `plan_viz_execute()` — 위 모두를 한 줄로 묶고 `path_quat` 유무에 따라 EE 경로 색을 바꿈

함수 *정의* 시점엔 setup 객체 (`node`, `move_client` 등) 를 lookup 하지 않으므로 객체가
아직 없어도 OK — 호출은 4 절(시나리오) 에서 일어난다.

### 2-1. 핵심에 필요한 import

In [3]:
import rclpy
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import (
    Constraints, MoveItErrorCodes,
    MotionPlanRequest, PlanningOptions, RobotState,
    OrientationConstraint,
)
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Point, Quaternion

### 2-2. `make_orientation_path_constraint()` — 경로 방향 제약 빌더

`goal_constraints` 의 OrientationConstraint 와 자료형은 같지만 의미가 다르다 —
**경로 내내** 이 방향을 (지정 tolerance 안에서) 유지해야 한다.

`tol_xy` 는 X / Y 축 (roll / pitch), `tol_z` 는 Z 축 (yaw) 허용 회전량 (rad).
yaw 자유도를 풀고 싶으면 `tol_z = math.pi` — 손목이 어떤 방향이든 허용.

In [4]:
def make_orientation_path_constraint(quat: Quaternion,
                                     tol_xy: float,
                                     tol_z: float) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    oc.orientation = quat
    oc.absolute_x_axis_tolerance = tol_xy
    oc.absolute_y_axis_tolerance = tol_xy
    oc.absolute_z_axis_tolerance = tol_z
    oc.weight = 1.0
    return oc

### 2-3. `send_move_goal()` + `plan_to_pose_goal()` — `path_quat` 옵셔널

ex07 의 plan-only wrapper 에 `path_quat` 인자 하나를 추가했다.
이 인자가 주어지면 `req.path_constraints` 에 OrientationConstraint 가 들어가고,
없으면 ex07 과 동일하게 동작한다.

**제약 하 플래닝은 비싸므로** 호출 측에서 `attempts` / `plan_time` 을 늘려 부르는 게 좋다
(시나리오에서는 15회 / 30초).

In [5]:
def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only,
        replan=not plan_only,
        replan_attempts=3 if not plan_only else 0,
    )
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory


def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       plan_time: float = 10.0, attempts: int = 5):
    req = make_plan_request(vel, acc, attempts=attempts, plan_time=plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj


def plan_to_pose_goal(pose, vel: float = 0.3, acc: float = 0.3,
                      plan_time: float = 10.0, attempts: int = 5,
                      path_quat=None, tol_xy: float = 0.3, tol_z: float = 3.14):
    '''path_quat 가 주어지면 경로 내내 그 방향이 유지되도록 path_constraints 추가.'''
    req = make_plan_request(vel, acc, attempts=attempts, plan_time=plan_time)
    if path_quat is not None:
        path_c = Constraints()
        path_c.orientation_constraints.append(
            make_orientation_path_constraint(path_quat, tol_xy, tol_z)
        )
        req.path_constraints = path_c
    goal_c = Constraints()
    goal_c.position_constraints.append(make_position_constraint(pose))
    goal_c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(goal_c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

### 2-4. `trajectory_to_ee_path()` — FK 로 EE 좌표 환산 (ex07 과 동일)

`MoveGroup` 의 곡선 trajectory 를 끝단 좌표 시퀀스로 변환. RViz 미리보기용.

In [6]:
def trajectory_to_ee_path(trajectory, max_points: int = 60):
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

### 2-5. `publish_ee_path()` + `execute_trajectory()`

EE 경로를 LINE_STRIP 으로 발행 (단계마다 덮어씀), 그리고 계획된 trajectory 만 실행.
EE 경로 색은 `color` 인자로 받아 — 시나리오에서 제약 하 / 없음 두 색을 달리 줄 수 있다.

In [7]:
def publish_ee_path(ee_points, color=None):
    if not ee_points:
        return
    if color is None:
        color = COLOR_EE_CONSTRAINED
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    _markers.markers = [m for m in _markers.markers
                        if (m.ns, m.id) != ('ee_path', 0)]
    _markers.markers.append(line)
    marker_pub.publish(_markers)


def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

### 2-6. `plan_viz_execute()` — 셋을 한 줄로 + 제약 색 분기

`path_quat` 가 주어지면 — path_constraints 적용 + EE 경로를 **청록**으로 그림.
없으면 — 일반 plan + EE 경로를 **회색**으로 그림.

시나리오에서 **같은 함수** 로 두 경우를 호출해 RViz 에서 색만 보고 차이를 비교 가능.

In [8]:
def plan_viz_execute(pose, vel: float = 0.3, label: str = '',
                     path_quat=None, tol_xy: float = 0.3, tol_z: float = 3.14,
                     attempts: int = 5, plan_time: float = 10.0) -> bool:
    constrained = path_quat is not None
    ok, traj = plan_to_pose_goal(
        pose, vel=vel, acc=vel,
        plan_time=plan_time, attempts=attempts,
        path_quat=path_quat, tol_xy=tol_xy, tol_z=tol_z,
    )
    if not ok or traj is None:
        node.get_logger().error(
            f'{label}: 계획 실패 (제약: {"있음" if constrained else "없음"})'
        )
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        color = COLOR_EE_CONSTRAINED if constrained else COLOR_EE_FREE
        publish_ee_path(pts, color=color)
    return execute_trajectory(traj)


def plan_viz_execute_joint(joint_values: dict, vel: float = 0.3,
                           label: str = '') -> bool:
    ok, traj = plan_to_joint_goal(joint_values, vel=vel, acc=vel)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts, color=COLOR_EE_FREE)
    return execute_trajectory(traj)

## 3. 핵심을 쓰기 위한 설정

위 핵심 함수들이 참조하는 객체와 보조 헬퍼.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 / `joint_states` 구독 / 마커 퍼블리셔 / FK 클라이언트
- 서버 / `joint_states` 준비 대기
- SRDF `ready` 자세
- Pose 헬퍼
- MoveGroup 빌딩블록 (Constraints / `MotionPlanRequest`)
- 마커 — 시작점 / 타깃 / 허용 오차 링 + EE 경로 색상 상수

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [9]:
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from visualization_msgs.msg import MarkerArray

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex08_constraints_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
fk_client      = node.create_client(GetPositionFK, 'compute_fk')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex08 노트북 노드 생성 완료 ===')

[INFO] [1778402866.426518907] [franka_ex08_constraints_demo]: === franka_ex08 노트북 노드 생성 완료 ===


True

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [10]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not fk_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_fk 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info(
        'move/execute action + compute_fk svc + /joint_states 준비됨'
    )

wait_for_ready()

[INFO] [1778402883.255678071] [franka_ex08_constraints_demo]: move/execute action + compute_fk svc + /joint_states 준비됨


### 3-3. SRDF 에서 `ready` 자세 읽어오기

In [11]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1778402884.915827394] [franka_ex08_constraints_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

### 3-4. Pose 헬퍼 — Euler ↔ Quaternion

In [12]:
import math
import tf_transformations
from geometry_msgs.msg import Pose

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`

In [13]:
from moveit_msgs.msg import (
    JointConstraint,
    PositionConstraint, BoundingVolume,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    '''goal_constraints 용 — pose 의 orientation 또는 Quaternion 직접 받음.'''
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

### 3-6. RViz 마커 — 시작점, 타깃, 허용 오차 링 + EE 경로 색

| 색상 | 의미 |
|---|---|
| 노랑 | 시작점 |
| 빨강 / 초록 / 파랑 | 타깃 A / B / C |
| 반투명 동심원 | 위치 허용 오차 (시각화용) |
| 청록 (`COLOR_EE_CONSTRAINED`) | path_constraints 적용된 plan 의 EE 경로 |
| 회색 (`COLOR_EE_FREE`) | 제약 없는 plan 의 EE 경로 (비교용) |

In [14]:
from std_msgs.msg import ColorRGBA

COLOR_TEXT          = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLOR_START         = ColorRGBA(r=1.0, g=0.8, b=0.0, a=1.0)
COLOR_EE_CONSTRAINED = ColorRGBA(r=0.0, g=0.9, b=0.9, a=0.95)  # 청록
COLOR_EE_FREE        = ColorRGBA(r=0.6, g=0.6, b=0.6, a=0.85)  # 회색
TARGET_COLORS = [
    ColorRGBA(r=0.9, g=0.2, b=0.2, a=1.0),   # A 빨강
    ColorRGBA(r=0.2, g=0.8, b=0.2, a=1.0),   # B 초록
    ColorRGBA(r=0.2, g=0.3, b=0.9, a=1.0),   # C 파랑
]

_markers = MarkerArray()

def publish_targets(start_pos, targets, tol_rad: float):
    '''시작점, 타깃 A/B/C, 허용 오차 링을 발행. ee_path 마커는 건드리지 않음.'''
    stamp = node.get_clock().now().to_msg()
    target_ns = {'pt', 'txt', 'ring'}
    _markers.markers = [m for m in _markers.markers if m.ns not in target_ns]

    sx, sy, sz = start_pos
    s_sphere = Marker()
    s_sphere.header.frame_id = REFERENCE_FRAME
    s_sphere.header.stamp = stamp
    s_sphere.ns = 'pt'
    s_sphere.id = 0
    s_sphere.type = Marker.SPHERE
    s_sphere.action = Marker.ADD
    s_sphere.pose.position = Point(x=sx, y=sy, z=sz)
    s_sphere.pose.orientation.w = 1.0
    s_sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
    s_sphere.color = COLOR_START
    s_text = Marker()
    s_text.header.frame_id = REFERENCE_FRAME
    s_text.header.stamp = stamp
    s_text.ns = 'txt'
    s_text.id = 0
    s_text.type = Marker.TEXT_VIEW_FACING
    s_text.action = Marker.ADD
    s_text.pose.position = Point(x=sx, y=sy, z=sz + 0.08)
    s_text.pose.orientation.w = 1.0
    s_text.scale.z = 0.04
    s_text.color = COLOR_TEXT
    s_text.text = 'Start'
    _markers.markers.extend([s_sphere, s_text])

    for i, tgt in enumerate(targets):
        tx, ty, tz = tgt['pos']
        c = TARGET_COLORS[i % len(TARGET_COLORS)]
        sphere = Marker()
        sphere.header.frame_id = REFERENCE_FRAME
        sphere.header.stamp = stamp
        sphere.ns = 'pt'
        sphere.id = i + 1
        sphere.type = Marker.SPHERE
        sphere.action = Marker.ADD
        sphere.pose.position = Point(x=tx, y=ty, z=tz)
        sphere.pose.orientation.w = 1.0
        sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
        sphere.color = c
        text = Marker()
        text.header.frame_id = REFERENCE_FRAME
        text.header.stamp = stamp
        text.ns = 'txt'
        text.id = i + 1
        text.type = Marker.TEXT_VIEW_FACING
        text.action = Marker.ADD
        text.pose.position = Point(x=tx, y=ty, z=tz + 0.08)
        text.pose.orientation.w = 1.0
        text.scale.z = 0.04
        text.color = COLOR_TEXT
        text.text = tgt['label']
        ring = Marker()
        ring.header.frame_id = REFERENCE_FRAME
        ring.header.stamp = stamp
        ring.ns = 'ring'
        ring.id = i
        ring.type = Marker.CYLINDER
        ring.action = Marker.ADD
        ring.pose.position = Point(x=tx, y=ty, z=tz)
        ring.pose.orientation.w = 1.0
        ring.scale.x = tol_rad * 0.4
        ring.scale.y = tol_rad * 0.4
        ring.scale.z = 0.005
        ring.color = ColorRGBA(r=c.r, g=c.g, b=c.b, a=0.25)
        _markers.markers.extend([sphere, text, ring])

    marker_pub.publish(_markers)

## 4. 시나리오 — 그리퍼 아래 방향 유지하고 A / B / C 순회

시작점 `(0.45, 0.00, 0.55)` 에서 그리퍼는 아래 (`roll=π`).
타깃 A / B / C 는 시작점 주변 좌우/위쪽. 모든 이동은 **path_constraint 하** 에서.
마지막에 같은 B 위치로 **제약 없이** 한 번 더 가서 RViz 에서 EE 경로 색을 비교한다.

### 4-1. ready 자세 + 시작점 / 타깃 / 제약 정의

In [15]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
time.sleep(1.0)

start_pos = (0.45, 0.00, 0.55)
start_pose = make_pose(*start_pos, math.pi, 0.0, 0.0)

down_quaternion = euler_to_quaternion(math.pi, 0.0, 0.0)  # 그리퍼 아래
TOL = 0.3   # X/Y 회전 허용 ±17°
TOL_Z = math.pi  # yaw 자유

targets = [
    {'label': 'A', 'pos': (0.45, -0.15, 0.45)},
    {'label': 'B', 'pos': (0.45,  0.15, 0.45)},
    {'label': 'C', 'pos': (0.40,  0.00, 0.65)},
]

publish_targets(start_pos, targets, TOL)
node.get_logger().info(
    f'방향 제약: 아래(roll=π), 허용 ±{math.degrees(TOL):.0f}°, yaw 자유'
)

[INFO] [1778402891.027152367] [franka_ex08_constraints_demo]: --- ready 자세로 초기 이동 ---
[INFO] [1778402892.351790298] [franka_ex08_constraints_demo]: 방향 제약: 아래(roll=π), 허용 ±17°, yaw 자유


True

### 4-2. 시작 위치로 (제약 없이) 이동

In [16]:
node.get_logger().info('--- 시작 위치로 이동 (제약 없이) ---')
plan_viz_execute(start_pose, label='Start')
time.sleep(1.0)

[INFO] [1778402895.807916084] [franka_ex08_constraints_demo]: --- 시작 위치로 이동 (제약 없이) ---


### 4-3. 타깃 A — 경로 제약 하 이동

In [17]:
tgt = targets[0]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 (청록 EE 경로) ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'],
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back to Start',
)
time.sleep(0.5)

[INFO] [1778402900.281117077] [franka_ex08_constraints_demo]: --- A 로 제약 이동 (청록 EE 경로) ---
[INFO] [1778402901.804217406] [franka_ex08_constraints_demo]:   결과: 성공


### 4-4. 타깃 B — 경로 제약 하 이동

In [18]:
tgt = targets[1]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 (청록 EE 경로) ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'],
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back to Start',
)
time.sleep(0.5)

[INFO] [1778402907.967277650] [franka_ex08_constraints_demo]: --- B 로 제약 이동 (청록 EE 경로) ---
[INFO] [1778402909.394580476] [franka_ex08_constraints_demo]:   결과: 성공


### 4-5. 타깃 C — 경로 제약 하 이동

In [19]:
tgt = targets[2]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 (청록 EE 경로) ---")
ok = plan_viz_execute(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label=tgt['label'],
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

plan_viz_execute(
    start_pose,
    path_quat=down_quaternion, tol_xy=TOL, tol_z=TOL_Z,
    attempts=15, plan_time=30.0, vel=0.2,
    label='Back to Start',
)
time.sleep(0.5)

[INFO] [1778402913.135171841] [franka_ex08_constraints_demo]: --- C 로 제약 이동 (청록 EE 경로) ---
[INFO] [1778402914.476842492] [franka_ex08_constraints_demo]:   결과: 성공


### 4-6. 비교 — 같은 B 로 **제약 없이** 이동

`path_quat` 인자를 안 넘기면 ex07 과 동일한 일반 plan. EE 경로가 **회색** 으로 그려진다.
RViz 에서 청록(제약 하) vs 회색(제약 없음) 경로 모양 차이를 비교 — 제약 없으면
손목이 휙 회전하는 trajectory 가 나올 수 있다.

In [20]:
node.get_logger().info('--- B 로 제약 없이 이동 (회색 EE 경로) ---')
plan_viz_execute(
    make_pose(*targets[1]['pos'], math.pi, 0.0, 0.0),
    label='B (no constraint)',
)
time.sleep(1.0)

plan_viz_execute(start_pose, label='Back to Start')
time.sleep(0.5)

[INFO] [1778402918.433618847] [franka_ex08_constraints_demo]: --- B 로 제약 없이 이동 (회색 EE 경로) ---


### 4-7. ready 복귀

In [21]:
node.get_logger().info('--- ready 복귀 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
node.get_logger().info('=== franka_ex08 완료! ===')

[INFO] [1778402922.844619831] [franka_ex08_constraints_demo]: --- ready 복귀 ---
[INFO] [1778402924.672642819] [franka_ex08_constraints_demo]: === franka_ex08 완료! ===


True

## 5. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass